# ISIC 2024: TabM Decoupled Tabular Hyperparameter Tuning with OptunaThis notebook implements **Stage 1 (Tabular-Only Optimization)** of the decoupled hyperparameter tuning workflow for the ISIC 2024 Challenge.### Cost Reduction & Speed Optimization Strategies:1. **RAM-Cached Data Pipeline**: All data loading, feature engineering, fold splitting, preprocessing, and bin computation are performed **once** and cached in RAM. Trials only create models and train — no CSV re-reads.2. **Dual Fold Evaluation (`fold_indices=[0, 1]`)**: Tuned on Fold 0 and Fold 1 to reduce noise and variance of the validation score, while remaining 2.5x faster than a full 5-fold evaluation.3. **Optuna Pruning (`MedianPruner`)**: Active pruning checks validation pAUC after each epoch. Sub-optimal trials are terminated early after epoch 15 if performance falls below the median.4. **Cap Max Epochs (`epochs=50`)**: Capped at 50 epochs, as the majority of the performance gains occur early.5. **Pre-Engineered Features**: Features are engineered once outside the trial loops.

In [ ]:
# =========================# CELL 1: SETUP ENVIRONMENT# =========================import sys, warningsfrom pathlib import Pathwarnings.filterwarnings('ignore', category=UserWarning, module='rtdl_num_embeddings')warnings.filterwarnings('ignore', category=FutureWarning)# Set project root path (edit as needed)CODE_DATASET_ROOT = '/kaggle/input/mid-fusion-lite' if not Path(CODE_DATASET_ROOT).exists():    CODE_DATASET_ROOT = '.'if CODE_DATASET_ROOT not in sys.path:    sys.path.insert(0, CODE_DATASET_ROOT)# Offline / Online dependencies installationoffline_dir = Path('/kaggle/input/tabm-kaggle-wheels')if offline_dir.exists():    print('Installing dependencies offline...')    !pip install /kaggle/input/tabm-kaggle-wheels/tabm-0.0.3-py3-none-any.whl /kaggle/input/tabm-kaggle-wheels/rtdl_num_embeddings-0.0.12-py3-none-any.whl --no-deps -qelse:    print('Installing dependencies online...')    !pip install -q tabm rtdl_num_embeddings optunaprint('Setup complete.')

In [ ]:
# =========================# CELL 2: DIRECTORIES & DATA PATHS# =========================import osis_kaggle = Path('/kaggle/input').exists()if is_kaggle:    INPUT_DIR = '/kaggle/input/isic-2024-challenge'    OUTPUT_DIR = '/kaggle/working/optuna_outputs'else:    INPUT_DIR = './archive'    OUTPUT_DIR = './optuna_outputs'os.makedirs(OUTPUT_DIR, exist_ok=True)TRAIN_METADATA_PATH = os.path.join(INPUT_DIR, 'train-metadata.csv')TEST_METADATA_PATH = os.path.join(INPUT_DIR, 'test-metadata.csv')SAMPLE_SUBMISSION_PATH = os.path.join(INPUT_DIR, 'sample_submission.csv')TRAIN_HDF5_PATH = os.path.join(INPUT_DIR, 'train-image.hdf5')TEST_HDF5_PATH = os.path.join(INPUT_DIR, 'test-image.hdf5')print(f'Input directory: {INPUT_DIR}')print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# =============================================================================# CELL 3: LOAD DATA, ENGINEER FEATURES, AND CACHE IN RAM# =============================================================================# All heavy I/O and feature engineering happens here — ONCE for the entire study.import gcimport timeimport numpy as npimport pandas as pdimport torchfrom sklearn.model_selection import StratifiedGroupKFoldimport config as config_modulefrom config import get_preset_config, BASE_NUMERIC_COLSfrom tabular_engineering_02 import add_feature_engineeringfrom tabular_preprocessor_04 import FoldTabularPreprocessor, select_available_columnsfrom data import HDF5ImageTabularDataset, make_balanced_samplerfrom torch.utils.data import DataLoadert0 = time.time()print('=' * 60)print('PHASE 1: One-time data loading & preprocessing')print('=' * 60)# ── 1) Read CSV ──────────────────────────────────────────────────────────────print('\n[1/5] Reading CSV files...')train_df = pd.read_csv(TRAIN_METADATA_PATH, low_memory=True)test_df = pd.read_csv(TEST_METADATA_PATH, low_memory=True)sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)print(f'  Train: {train_df.shape}, Test: {test_df.shape}')# ── 2) Feature Engineering ───────────────────────────────────────────────────print('[2/5] Running feature engineering (one-time cost)...')train_df, test_df = add_feature_engineering(    train_df=train_df,    test_df=test_df,    group_col='patient_id',    categorical_cols=config_module.DEFAULT_CATEGORICAL_COLS,)print(f'  Engineered columns: {len(train_df.columns)}')# ── 3) Build base config & select columns ────────────────────────────────────print('[3/5] Building config & selecting columns...')BASE_CONFIG = get_preset_config('quick').with_updates(    use_feature_engineering=False,    n_splits=5,    fold_indices=[0, 1],    epochs=50,    patience=10,    modality_mode='tabular_only',    use_tabm_tabular=True,    tabm_arch_type='tabm',    save_checkpoints=False,    show_predict_progress=False,    num_workers=0,    train_hdf5_path=TRAIN_HDF5_PATH,    test_hdf5_path=TEST_HDF5_PATH,    sample_submission_path=SAMPLE_SUBMISSION_PATH,)numeric_cols, categorical_cols = select_available_columns(    train_df=train_df,    test_df=test_df,    numeric_candidates=BASE_CONFIG.numeric_cols,    categorical_candidates=BASE_CONFIG.categorical_cols,)print(f'  Numeric: {len(numeric_cols)}, Categorical: {len(categorical_cols)}')# ── 4) Compute CV folds ─────────────────────────────────────────────────────print('[4/5] Computing CV folds...')y_all = train_df[BASE_CONFIG.target_col].astype(np.int64).to_numpy()groups_all = train_df[BASE_CONFIG.group_col].to_numpy()splitter = StratifiedGroupKFold(    n_splits=BASE_CONFIG.n_splits,    shuffle=True,    random_state=BASE_CONFIG.seed,)ALL_FOLDS = list(splitter.split(train_df, y_all, groups_all))SELECTED_FOLDS = BASE_CONFIG.fold_indices or list(range(BASE_CONFIG.n_splits))print(f'  Selected folds: {SELECTED_FOLDS}')# ── 5) Pre-compute per-fold preprocessor outputs ────────────────────────────print('[5/5] Pre-computing fold preprocessors, transforms & bins...')# Cache structure: dict[fold_index] -> {x_train, x_valid, y_train, y_valid, ...}FOLD_CACHE = {}for fold_idx in SELECTED_FOLDS:    train_idx, valid_idx = ALL_FOLDS[fold_idx]    fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)    fold_valid_df = train_df.iloc[valid_idx].reset_index(drop=True)        # Fit preprocessor (StandardScaler + OneHotEncoder + QuantileTransformer)    preprocessor = FoldTabularPreprocessor(        numeric_cols=numeric_cols,        categorical_cols=categorical_cols,        use_quantile_transform=True,        quantile_noise_std=float(BASE_CONFIG.quantile_noise_std),        quantile_random_state=BASE_CONFIG.seed + fold_idx + 1,        use_null_importance_filter=False,        target_col=BASE_CONFIG.target_col,    )    preprocessor.fit(fold_train_df)        x_train = preprocessor.transform(fold_train_df)    x_valid = preprocessor.transform(fold_valid_df)    y_train = fold_train_df[BASE_CONFIG.target_col].astype(np.float32).to_numpy()    y_valid = fold_valid_df[BASE_CONFIG.target_col].astype(np.float32).to_numpy()        # IDs for dataset construction    ids_train = fold_train_df[BASE_CONFIG.id_col].astype(str).to_numpy()    ids_valid = fold_valid_df[BASE_CONFIG.id_col].astype(str).to_numpy()        # Hard negative weights    if 'tbp_lv_nevi_confidence' in fold_train_df.columns:        nevi_conf = fold_train_df['tbp_lv_nevi_confidence'].fillna(50.0).to_numpy()        hard_neg_w = 0.1 + 0.9 * (((100.0 - nevi_conf) / 100.0) ** 2.0)    else:        hard_neg_w = None        # Pos weight    n_pos = max(float(y_train.sum()), 1.0)    n_neg = max(float(len(y_train) - y_train.sum()), 1.0)    pos_weight = np.sqrt(n_neg / n_pos)        # Sampler    sampler = make_balanced_sampler(        targets=y_train,        negative_ratio=BASE_CONFIG.negative_ratio,        seed=BASE_CONFIG.seed + fold_idx,        hard_negative_weights=hard_neg_w,    )        FOLD_CACHE[fold_idx] = {        'x_train': x_train,        'x_valid': x_valid,        'y_train': y_train,        'y_valid': y_valid,        'ids_train': ids_train,        'ids_valid': ids_valid,        'valid_idx': valid_idx,        'pos_weight': pos_weight,        'sampler': sampler,    }    print(f'  Fold {fold_idx}: train={len(y_train)}, valid={len(y_valid)}, n_features={x_train.shape[1]}')# Cache global infoN_TAB_FEATURES = FOLD_CACHE[SELECTED_FOLDS[0]]['x_train'].shape[1]# Free the large DataFrames we no longer need for trialsdel fold_train_df, fold_valid_dfgc.collect()elapsed = time.time() - t0print(f'\n✅ Data preparation complete in {elapsed:.1f}s. All cached in RAM.')print(f'   Each trial will skip CSV reads, feature engineering, fold splitting, and preprocessing.')

In [ ]:
# =============================================================================# CELL 4: DEFINE OPTUNA OBJECTIVE FUNCTION (USES RAM CACHE)# =============================================================================import optunaimport rtdl_num_embeddingsfrom model import MidFusionLiteModelfrom train import fit_one_fold, partial_auc_scorefrom runner import _initialize_lazy_parametersdef _compute_bins_for_trial(x_train: np.ndarray, n_bins: int):    """Compute PLE bins with margin widening (matches runner.py logic)."""    x_tensor = torch.tensor(x_train, dtype=torch.float32)    raw_bins = rtdl_num_embeddings.compute_bins(x_tensor, n_bins=n_bins)    bins = []    for b in raw_bins:        b_wide = b.clone()        if len(b_wide) > 1:            span = b_wide[-1] - b_wide[0]            margin = max(1e-3, float(0.1 * span))            b_wide[0] -= margin            b_wide[-1] += margin        bins.append(b_wide)    return binsdef objective(trial):    # ── Sample hyperparameters ───────────────────────────────────────────────    n_bins = trial.suggest_int('n_bins', 48, 160, step=16)    d_embedding = trial.suggest_int('d_embedding', 16, 48, step=8)    tabm_k = trial.suggest_int('tabm_k', 8, 32, step=8)    tabm_n_blocks = trial.suggest_int('tabm_n_blocks', 2, 5)    tabm_d_block = trial.suggest_int('tabm_d_block', 128, 512, step=64)    tab_dropout = trial.suggest_float('tab_dropout', 0.05, 0.30)    lr = trial.suggest_float('lr', 1e-4, 1e-3, log=True)    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)        print(f'\n[Trial {trial.number}] Parameters: {trial.params}')        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')        # Pruning callback (only report/prune on the first fold to avoid step collisions in CV)    def prune_callback(epoch, score):        if fold_idx == SELECTED_FOLDS[0]:            trial.report(score, step=epoch)            if trial.should_prune():                raise optuna.exceptions.TrialPruned()        # ── Train across selected folds ──────────────────────────────────────────    oof_pred = np.zeros(len(y_all), dtype=np.float32)    oof_mask = np.zeros(len(y_all), dtype=bool)    fold_scores = []        for fold_idx in SELECTED_FOLDS:        cache = FOLD_CACHE[fold_idx]                # Compute bins per-trial (n_bins varies between trials)        bins = _compute_bins_for_trial(cache['x_train'], n_bins)                # Build lightweight tabular-only datasets (no HDF5 reads)        train_ds = HDF5ImageTabularDataset(            ids=cache['ids_train'],            tabular=cache['x_train'],            hdf5_path=TRAIN_HDF5_PATH,            targets=cache['y_train'],            transforms=None,            use_shades_of_gray=False,            modality_mode='tabular_only',            image_size=BASE_CONFIG.image_size,        )        valid_ds = HDF5ImageTabularDataset(            ids=cache['ids_valid'],            tabular=cache['x_valid'],            hdf5_path=TRAIN_HDF5_PATH,            targets=cache['y_valid'],            transforms=None,            use_shades_of_gray=False,            modality_mode='tabular_only',            image_size=BASE_CONFIG.image_size,        )                train_loader = DataLoader(            train_ds,            batch_size=BASE_CONFIG.train_batch_size,            sampler=cache['sampler'],            num_workers=0,            pin_memory=(device.type == 'cuda'),            drop_last=False,        )        valid_loader = DataLoader(            valid_ds,            batch_size=BASE_CONFIG.valid_batch_size,            shuffle=False,            num_workers=0,            pin_memory=(device.type == 'cuda'),            drop_last=False,        )                # Build model        model = MidFusionLiteModel(            backbone_name=BASE_CONFIG.backbone_name,            n_tab_features=N_TAB_FEATURES,            use_tabm_tabular=True,            tab_hidden_dim=BASE_CONFIG.tab_hidden_dim,            tabm_arch_type='tabm',            tabm_k=tabm_k,            tabm_n_blocks=tabm_n_blocks,            tabm_d_block=tabm_d_block,            fusion_hidden_dim=BASE_CONFIG.fusion_hidden_dim,            fusion_mode=BASE_CONFIG.fusion_mode,            cross_attention_heads=BASE_CONFIG.cross_attention_heads,            cross_attention_dropout=BASE_CONFIG.cross_attention_dropout,            tab_dropout=tab_dropout,            fusion_dropout=BASE_CONFIG.fusion_dropout,            train_image_backbone=False,            use_pretrained_backbone=False,            modality_dropout_prob=0.0,            modality_mode='tabular_only',            bins=bins,            d_embedding=d_embedding,            init_bias=None,        ).to(device)                _initialize_lazy_parameters(model, device, BASE_CONFIG.image_size, N_TAB_FEATURES)                optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)                best_state, history, best_score = fit_one_fold(            model=model,            train_loader=train_loader,            valid_loader=valid_loader,            optimizer=optimizer,            scheduler=None,            device=device,            epochs=BASE_CONFIG.epochs,            patience=BASE_CONFIG.patience,            pos_weight=cache['pos_weight'],            use_amp=(BASE_CONFIG.use_amp and device.type == 'cuda'),            max_train_steps_per_epoch=BASE_CONFIG.max_train_steps_per_epoch,            max_valid_steps=BASE_CONFIG.max_valid_steps,            aux_loss_image_weight=0.0,            aux_loss_tab_weight=0.0,            gradient_clipping_norm=BASE_CONFIG.gradient_clipping_norm,            label_smoothing=BASE_CONFIG.label_smoothing,            epoch_callback=prune_callback,        )                fold_scores.append(best_score)        oof_pred[cache['valid_idx']] = best_score  # Use best score as proxy        oof_mask[cache['valid_idx']] = True                # Cleanup        del model, optimizer, bins, train_ds, valid_ds, train_loader, valid_loader        if device.type == 'cuda':            torch.cuda.empty_cache()        cv_score = float(np.mean(fold_scores))    print(f'[Trial {trial.number}] Complete. Mean best pAUC = {cv_score:.6f}')    return cv_scoreprint('Objective function defined (RAM-cached mode).')

In [ ]:
# =========================# CELL 5: RUN OPTUNA STUDY WITH MEDIAN PRUNER# =========================optuna.logging.set_verbosity(optuna.logging.WARNING)# Use MedianPruner to prune trials after 15 warmup epochspruner = optuna.pruners.MedianPruner(n_warmup_steps=15)study = optuna.create_study(direction='maximize', pruner=pruner)study.optimize(objective, n_trials=80, timeout=14400) # Max 4 hours / 80 trialsprint('\n' + '=' * 60)print('OPTUNA OPTIMIZATION COMPLETE')print('=' * 60)print(f'Best CV pAUC Score: {study.best_value:.6f}')print('Best Parameters:')for k, v in study.best_params.items():    print(f'  {k}: {v}')

In [ ]:
# =========================# CELL 6: ANALYZE AND SAVE RESULTS# =========================trials_df = study.trials_dataframe()csv_path = os.path.join(OUTPUT_DIR, 'optuna_tabm_trials.csv')trials_df.to_csv(csv_path, index=False)print(f'Saved trials results to: {csv_path}')try:    import matplotlib.pyplot as plt    fig1 = optuna.visualization.plot_optimization_history(study)    fig1.show()    fig2 = optuna.visualization.plot_param_importances(study)    fig2.show()except Exception as e:    print('Could not display visualizations:', e)